# Incident Response Runbook: Reconnaissance - Gather Victim Information

**Tactic:** Reconnaissance
**Technique:** Gather Victim Information (T1592)
**Severity:** MEDIUM

## Overview

This runbook provides step-by-step incident response procedures for Gather Victim Information activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add the project root to the Python path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()

# Query Splunk for victim information gathering indicators
print(f"\n[QUERY] Searching Splunk for victim information gathering indicators...")
splunk_query = '''
index=* (sourcetype=WinEventLog:Security EventCode=4625 OR EventCode=4648)
OR (sourcetype=linux_secure "invalid user" OR "Failed password" OR "authentication failure")
OR (sourcetype=access_combined status=404 | stats count by clientip | where count > 50)
OR (sourcetype=dns "NXDOMAIN" OR "SERVFAIL" | stats count by src_ip | where count > 20)
OR (sourcetype=WinEventLog:Security EventCode=4768 OR EventCode=4771)
OR (sourcetype=network "SMB" OR "RDP" OR "SSH" dest_port!="22" dest_port!="3389" dest_port!="445")
| eval source_ip=coalesce(src_ip, clientip, SourceAddress, src)
| eval target_info=coalesce(dest_ip, dest, destination, target)
| eval recon_type=case(match(_raw, "password"), "credential_stuffing", match(_raw, "404"), "web_crawling", match(_raw, "NXDOMAIN"), "dns_enumeration", match(_raw, "SMB|RDP|SSH"), "service_probing", true(), "unknown")
| stats count by source_ip, target_info, recon_type, _time
| where count > 10
'''

try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-24h")
    print(f"   Found {len(splunk_results)} potential victim information gathering events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

# Extract affected systems and reconnaissance activities
affected_systems = []
recon_activities = []
splunk_indicators = []

for event in splunk_results:
    system_info = {
        'source_ip': event.get('source_ip', 'unknown'),
        'target_info': event.get('target_info', ''),
        'recon_type': event.get('recon_type', 'unknown'),
        'attempt_count': event.get('count', 0),
        'last_seen': event.get('_time', detection_time)
    }
    affected_systems.append(system_info)

    # Extract indicators
    if event.get('source_ip') and event['source_ip'] != 'unknown':
        splunk_indicators.append({
            'type': 'ip',
            'value': event.get('source_ip'),
            'context': f"Information gathering from {event.get('source_ip')} targeting {event.get('target_info', 'multiple targets')}"
        })

# Query CrowdStrike for reconnaissance detections
print(f"\n[QUERY] Checking CrowdStrike for reconnaissance detections...")
try:
    cs_detections = crowdstrike.get_detections(
        filter="technique:'Gather Victim Network Information'",
        start_time="-24h"
    )
    cs_recon_sources = []
    for detection in cs_detections:
        recon_info = {
            'source_ip': detection.get('source_ip', ''),
            'hostname': detection.get('hostname', ''),
            'detection_id': detection.get('detection_id', ''),
            'technique': detection.get('technique', ''),
            'severity': detection.get('severity', 0),
            'recon_details': detection.get('recon_details', {})
        }
        cs_recon_sources.append(recon_info)

        # Merge with Splunk data
        existing_system = next((s for s in affected_systems if s['source_ip'] == recon_info['source_ip']), None)
        if existing_system:
            existing_system.update(recon_info)
        else:
            affected_systems.append(recon_info)

    print(f"   Found {len(cs_recon_sources)} CrowdStrike reconnaissance detections")
except Exception as e:
    print(f"   CrowdStrike query failed: {e}")
    cs_recon_sources = []

# Enrich with threat intelligence from MISP
print(f"\n[ENRICHMENT] Checking MISP for related threat intelligence...")
misp_results = []
try:
    for indicator in splunk_indicators:
        misp_search = misp.search_iocs(indicator['value'])
        if misp_search:
            misp_results.extend(misp_search)
            print(f"   Found threat intelligence for {indicator['value']}")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    incident_data = {
        'title': f'Victim Information Gathering Incident - {len(affected_systems)} reconnaissance sources',
        'description': f'Detected victim information gathering activities from {len(affected_systems)} sources',
        'severity': 'MEDIUM',
        'tactic': 'Reconnaissance',
        'technique': 'Gather Victim Network Information (T1590)',
        'affected_systems': affected_systems,
        'indicators': splunk_indicators,
        'threat_intel': misp_results,
        'detection_time': detection_time
    }
    incident_id = iris.create_case(incident_data)
    print(f"   Created IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"TEMP-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n Detection complete:")
print(f"  - {len(affected_systems)} reconnaissance sources identified")
print(f"  - {len(splunk_indicators)} information gathering indicators found")
print(f"  - {len(misp_results)} threat intelligence matches")
print(f"  - IRIS case {incident_id} created")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
isolated_hosts = []
blocked_ips = []
disabled_services = []

# Isolate affected systems
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            isolation_result = crowdstrike.isolate_host(system['device_id'])
            if isolation_result.get('status') == 'isolated':
                isolated_hosts.append(system.get('hostname', system.get('source_ip', 'unknown')))
                print(f"   Isolated host: {system.get('hostname', system.get('source_ip', 'unknown'))}")
    except Exception as e:
        print(f"   Host isolation failed for {system.get('hostname', system.get('source_ip', 'unknown'))}: {e}")

# Block reconnaissance source IPs
print(f"\n[ACTION] Blocking reconnaissance source IPs...")
unique_ips = set()
for system in affected_systems:
    if system.get('source_ip') and system['source_ip'] != 'unknown':
        unique_ips.add(system['source_ip'])

for ip in unique_ips:
    try:
        block_result = shuffle.block_ip_address(ip)
        if block_result.get('status') == 'blocked':
            blocked_ips.append(ip)
            print(f"   Blocked reconnaissance IP: {ip}")
    except Exception as e:
        print(f"   IP blocking failed for {ip}: {e}")

# Disable vulnerable services
print(f"\n[ACTION] Disabling vulnerable services...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            service_disable = crowdstrike.disable_vulnerable_services(system['device_id'])
            if service_disable.get('status') == 'disabled':
                disabled_services.extend(service_disable.get('disabled_services', []))
                print(f"   Disabled {len(service_disable.get('disabled_services', []))} vulnerable services on {system.get('hostname', system.get('source_ip', 'unknown'))}")
    except Exception as e:
        print(f"   Service disable failed for {system.get('hostname', system.get('source_ip', 'unknown'))}: {e}")

# Enable enhanced monitoring
print(f"\n[ACTION] Enabling enhanced monitoring...")
monitoring_rules = []
try:
    # Enable CrowdStrike reconnaissance monitoring
    cs_monitoring = crowdstrike.enable_enhanced_monitoring('reconnaissance')
    if cs_monitoring.get('status') == 'enabled':
        monitoring_rules.append('CrowdStrike reconnaissance monitoring')
        print(f"   Enabled CrowdStrike reconnaissance monitoring")

    # Enable Splunk information gathering correlation rules
    splunk_monitoring = splunk.enable_correlation_rule('information_gathering')
    if splunk_monitoring.get('status') == 'enabled':
        monitoring_rules.append('Splunk information gathering correlation')
        print(f"   Enabled Splunk information gathering correlation rules")
except Exception as e:
    print(f"   Enhanced monitoring setup failed: {e}")

# Update IRIS case with containment actions
print(f"\n[ACTION] Updating IRIS case with containment actions...")
try:
    containment_summary = {
        'isolated_hosts': len(isolated_hosts),
        'blocked_ips': len(blocked_ips),
        'disabled_services': len(disabled_services),
        'monitoring_enabled': len(monitoring_rules),
        'containment_time': containment_time
    }
    iris.update_case(incident_id, 'containment_complete', containment_summary)
    print(f"   Updated IRIS case {incident_id} with containment results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_hosts)} hosts isolated")
print(f"  - {len(blocked_ips)} reconnaissance IPs blocked")
print(f"  - {len(disabled_services)} vulnerable services disabled")
print(f"  - {len(monitoring_rules)} enhanced monitoring rules enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

removed_recon_artifacts = []
patched_systems = []
cleaned_logs = []

# Remove reconnaissance artifacts
print(f"\n[ACTION] Removing reconnaissance artifacts...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            artifact_removal = crowdstrike.remove_recon_artifacts(system['device_id'])
            if artifact_removal.get('status') == 'removed':
                removed_recon_artifacts.extend(artifact_removal.get('removed_files', []))
                print(f"   Removed {len(artifact_removal.get('removed_files', []))} reconnaissance artifacts from {system.get('hostname', system.get('source_ip', 'unknown'))}")
    except Exception as e:
        print(f"   Recon artifact removal failed for {system.get('hostname', system.get('source_ip', 'unknown'))}: {e}")

# Patch systems to prevent information gathering
print(f"\n[ACTION] Patching systems to prevent information gathering...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            patch_result = crowdstrike.apply_security_patches(system['device_id'])
            if patch_result.get('status') == 'patched':
                patched_systems.append(system.get('hostname', system.get('source_ip', 'unknown')))
                print(f"   Applied security patches to {system.get('hostname', system.get('source_ip', 'unknown'))}")
    except Exception as e:
        print(f"   Patching failed for {system.get('hostname', system.get('source_ip', 'unknown'))}: {e}")

# Clean reconnaissance logs
print(f"\n[ACTION] Cleaning reconnaissance logs...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            log_cleaning = crowdstrike.clean_reconnaissance_logs(system['device_id'])
            if log_cleaning.get('status') == 'cleaned':
                cleaned_logs.append(system.get('hostname', system.get('source_ip', 'unknown')))
                print(f"   Cleaned reconnaissance logs on {system.get('hostname', system.get('source_ip', 'unknown'))}")
    except Exception as e:
        print(f"   Log cleaning failed for {system.get('hostname', system.get('source_ip', 'unknown'))}: {e}")

# Harden network perimeter
print(f"\n[ACTION] Hardening network perimeter...")
try:
    perimeter_hardening = shuffle.harden_network_perimeter()
    if perimeter_hardening.get('status') == 'hardened':
        print(f"   Hardened network perimeter against reconnaissance")
except Exception as e:
    print(f"   Network perimeter hardening failed: {e}")

# Verify threat removal
print(f"\n[ACTION] Verifying threat removal...")
verification_results = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            verify_result = crowdstrike.verify_recon_removal(system['device_id'])
            verification_results.append({
                'system': system.get('hostname', system.get('source_ip', 'unknown')),
                'status': 'clean' if verify_result.get('recon_detected', True) == False else 'threats_remaining',
                'remaining_indicators': verify_result.get('remaining_indicators', 0)
            })
            status = "" if verify_result.get('recon_detected', True) == False else ""
            print(f"  {status} Verification for {system.get('hostname', system.get('source_ip', 'unknown'))}: {verify_result.get('remaining_indicators', 0)} reconnaissance indicators remaining")
    except Exception as e:
        print(f"   Verification failed for {system.get('hostname', system.get('source_ip', 'unknown'))}: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(removed_recon_artifacts)} reconnaissance artifacts removed")
print(f"  - {len(patched_systems)} systems patched")
print(f"  - {len(cleaned_logs)} systems had logs cleaned")
print(f"  - {len([v for v in verification_results if v['status'] == 'clean'])} systems verified clean")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

restored_systems = []
validated_functions = []
monitoring_restored = []

# Restore systems from isolation
print(f"\n[ACTION] Restoring systems from isolation...")
for system in affected_systems:
    try:
        if 'device_id' in system and system.get('hostname'):
            restore_result = crowdstrike.restore_network_connectivity(system['device_id'])
            if restore_result.get('status') == 'restored':
                restored_systems.append(system['hostname'])
                print(f"   Restored connectivity for: {system['hostname']}")
    except Exception as e:
        print(f"   System restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Validate system functionality
print(f"\n[ACTION] Validating system functionality...")
functional_tests = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            test_result = crowdstrike.perform_functional_testing(system['device_id'])
            functional_tests.append({
                'system': system.get('hostname', system.get('source_ip', 'unknown')),
                'tests_passed': test_result.get('tests_passed', 0),
                'total_tests': test_result.get('total_tests', 0)
            })
            pass_rate = test_result.get('tests_passed', 0) / max(test_result.get('total_tests', 1), 1) * 100
            status = "" if pass_rate >= 95 else ""
            print(f"  {status} Functional tests for {system.get('hostname', system.get('source_ip', 'unknown'))}: {test_result.get('tests_passed', 0)}/{test_result.get('total_tests', 0)} passed ({pass_rate:.1f}%)")
    except Exception as e:
        print(f"   Functional testing failed for {system.get('hostname', system.get('source_ip', 'unknown'))}: {e}")

# Restore monitoring capabilities
print(f"\n[ACTION] Restoring monitoring capabilities...")
try:
    monitoring_restore = splunk.restore_monitoring()
    if monitoring_restore.get('status') == 'restored':
        monitoring_restored.append('splunk_monitoring')
        print(f"   Restored Splunk monitoring capabilities")

    cs_monitoring_restore = crowdstrike.restore_monitoring()
    if cs_monitoring_restore.get('status') == 'restored':
        monitoring_restored.append('crowdstrike_monitoring')
        print(f"   Restored CrowdStrike monitoring capabilities")
except Exception as e:
    print(f"   Monitoring restoration failed: {e}")

# Monitor for residual issues
print(f"\n[ACTION] Establishing monitoring for residual issues...")
residual_monitoring = []
try:
    # Set up continuous monitoring
    monitoring_setup = splunk.setup_continuous_monitoring('reconnaissance')
    if monitoring_setup.get('status') == 'enabled':
        residual_monitoring.append('Splunk continuous monitoring')
        print(f"   Established continuous Splunk monitoring")

    cs_monitoring = crowdstrike.setup_residual_monitoring('reconnaissance')
    if cs_monitoring.get('status') == 'enabled':
        residual_monitoring.append('CrowdStrike residual monitoring')
        print(f"   Established CrowdStrike residual monitoring")
except Exception as e:
    print(f"   Residual monitoring setup failed: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored from isolation")
print(f"  - {len([f for f in functional_tests if f['tests_passed'] == f['total_tests']])} systems passed all functional tests")
print(f"  - {len(monitoring_restored)} monitoring systems restored")
print(f"  - {len(residual_monitoring)} residual monitoring systems established")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident
print("\n" + "=" * 60)
print("STEP 5: Post-Incident")
print("=" * 60)

# Update IRIS case with eradication results
print(f"\n[ACTION] Updating IRIS case with eradication results...")
try:
    eradication_summary = {
        'removed_recon_artifacts': len(removed_recon_artifacts),
        'patched_systems': len(patched_systems),
        'cleaned_logs': len(cleaned_logs),
        'verification_results': verification_results
    }
    iris.update_case(incident_id, 'eradication_complete', eradication_summary)
    print(f"   Updated IRIS case {incident_id} with eradication results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

# Update IRIS case with recovery results
print(f"\n[ACTION] Updating IRIS case with recovery results...")
try:
    recovery_summary = {
        'restored_systems': len(restored_systems),
        'functional_tests': functional_tests,
        'monitoring_restored': len(monitoring_restored),
        'residual_monitoring': len(residual_monitoring)
    }
    iris.update_case(incident_id, 'recovery_complete', recovery_summary)
    print(f"   Updated IRIS case {incident_id} with recovery results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

# Generate incident report
print(f"\n[ACTION] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'technique': 'Reconnaissance: Gather Victim Network Information',
        'detection_time': detection_time,
        'affected_systems': len(affected_systems),
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': datetime.now().isoformat(),
            'recovery': datetime.now().isoformat()
        },
        'actions_taken': {
            'detection': f"Found {len(affected_systems)} reconnaissance sources",
            'containment': f"Isolated {len(isolated_hosts)} hosts, blocked {len(blocked_ips)} IPs",
            'eradication': f"Removed {len(removed_recon_artifacts)} artifacts, patched {len(patched_systems)} systems",
            'recovery': f"Restored {len(restored_systems)} systems, validated {len([f for f in functional_tests if f['tests_passed'] == f['total_tests']])} functions"
        },
        'indicators': splunk_indicators,
        'threat_intel': misp_results,
        'lessons_learned': [
            "Implement stricter network reconnaissance detection",
            "Regular monitoring of authentication failures and unusual access patterns",
            "Enhanced threat intelligence integration for reconnaissance activities"
        ]
    }
    iris.generate_report(incident_id, incident_report)
    print(f"   Generated comprehensive incident report for case {incident_id}")
except Exception as e:
    print(f"   Report generation failed: {e}")

# Share IOCs with MISP
print(f"\n[ACTION] Sharing IOCs with MISP...")
try:
    misp_iocs = []
    for indicator in splunk_indicators:
        if indicator.get('type') in ['ip']:
            misp_iocs.append({
                'type': indicator['type'],
                'value': indicator['value'],
                'tags': ['victim-info-gathering', 'reconnaissance', 'incident-' + str(incident_id)]
            })

    if misp_iocs:
        misp.share_iocs(misp_iocs, f"Victim Information Gathering Incident {incident_id}")
        print(f"   Shared {len(misp_iocs)} IOCs with MISP community")
    else:
        print(f"   No new IOCs to share with MISP")
except Exception as e:
    print(f"   MISP IOC sharing failed: {e}")

# Close IRIS case
print(f"\n[ACTION] Closing IRIS case...")
try:
    iris.close_case(incident_id, 'resolved')
    print(f"   Closed IRIS case {incident_id} as resolved")
except Exception as e:
    print(f"   IRIS case closure failed: {e}")

# Final status summary
print(f"\n Post-incident activities complete:")
print(f"  - IRIS case {incident_id} updated and closed")
print(f"  - Incident report generated")
print(f"  - {len(misp_iocs) if 'misp_iocs' in locals() else 0} IOCs shared with threat intelligence community")
print(f"  - All 5 IR phases completed successfully")

# Calculate incident metrics
incident_duration = (datetime.now() - datetime.fromisoformat(detection_time)).total_seconds() / 3600
print(f"\n📊 Incident Metrics:")
print(f"  - Duration: {incident_duration:.2f} hours")
print(f"  - Systems affected: {len(affected_systems)}")
print(f"  - Containment time: {'< 1 hour' if incident_duration < 1 else f'{incident_duration:.1f} hours'}")
print(f"  - Eradication success: {len([v for v in verification_results if v['status'] == 'clean'])}/{len(affected_systems)} systems clean")
print(f"  - Recovery success: {len([f for f in functional_tests if f['tests_passed'] == f['total_tests']])}/{len(affected_systems)} systems fully functional")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
